[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imbishal7/CSC-428/blob/main/notebooks/CSC428_project.ipynb)

# CSC428 Final Project — ML Side-Channel Attack on AES (ASCAD)

Suwan Aryal & Kapil Sharma. We train classifiers to recover the Hamming weight of an AES key byte from power-trace measurements collected on an ATMega8515 microcontroller.

**Pipeline:**

1. Data downloading and loading
2. Data cleaning
3. Train / validation / test split
4. Define, train, validate, and test three models — Logistic Regression, MLP, CNN
5. Compare results

## Section 0 — Colab setup
Mount Drive, clone the repo (first run only), install dependencies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive
![ -d CSC-428 ] || git clone https://github.com/imbishal7/CSC-428.git
%cd CSC-428
!pip install -q -r requirements.txt
!nvidia-smi -L

## 1. Data downloading and loading

ASCAD is published by ANSSI as a single ~4.2 GB zip. The download script grabs it and extracts `ASCAD.h5` into `data/`; it skips work that's already done.

The file holds 60,000 power traces (50,000 profiling + 10,000 attack), each 700 samples long. Labels are the AES Sbox output `Sbox(plaintext[2] XOR key[2])`. We convert these to **Hamming weight** (9 classes, 0–8) since the proposal targets HW recovery.

In [ ]:
!python scripts/download_data.py

In [ ]:
import h5py
import numpy as np

HW_TABLE = np.array([bin(i).count('1') for i in range(256)], dtype=np.int64)

with h5py.File('data/ASCAD.h5', 'r') as f:
    X_train_full = np.array(f['Profiling_traces/traces'], dtype=np.float32)
    y_train_full = HW_TABLE[np.array(f['Profiling_traces/labels'], dtype=np.int64)]
    X_test       = np.array(f['Attack_traces/traces'],     dtype=np.float32)
    y_test       = HW_TABLE[np.array(f['Attack_traces/labels'], dtype=np.int64)]

print('profiling traces:', X_train_full.shape, '  labels:', y_train_full.shape)
print('attack traces:   ', X_test.shape,       '  labels:', y_test.shape)
print('classes:', np.unique(y_train_full))

## 2. Data cleaning

ASCAD is published already pre-aligned and free of missing values, so cleaning here is light: verify there are no NaN / infinite values, look at the class distribution (which is **not** balanced — the binomial distribution of 8-bit Hamming weights peaks at 4 and tapers to almost nothing at 0 and 8), and standardize the features so each of the 700 time points has zero mean and unit variance.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

assert not np.isnan(X_train_full).any() and not np.isinf(X_train_full).any(), 'NaN/Inf in profiling traces'
assert not np.isnan(X_test).any()       and not np.isinf(X_test).any(),       'NaN/Inf in attack traces'
print('no NaN / Inf')

counts = np.bincount(y_train_full, minlength=9)
print('class distribution (profiling):', dict(enumerate(counts)))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].bar(range(9), counts); ax[0].set(xlabel='Hamming weight', ylabel='count', title='Profiling label distribution')
ax[1].plot(X_train_full[0]);  ax[1].set(xlabel='time sample', ylabel='power', title='Example trace (first profiling sample)')
plt.tight_layout(); plt.show()

scaler = StandardScaler().fit(X_train_full)
X_train_full = scaler.transform(X_train_full).astype(np.float32)
X_test       = scaler.transform(X_test).astype(np.float32)
print('standardized:', X_train_full.shape, X_test.shape)

## 3. Train / validation / test split

ASCAD already separates *profiling* (model-development) traces from *attack* (held-out) traces — we use the attack set as our test set. We carve a 10% slice off the profiling traces for validation. Stratified so the rare HW=0 and HW=8 classes are represented in both halves.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.1, random_state=42, stratify=y_train_full,
)
print(f"train: {X_train.shape}  val: {X_val.shape}  test: {X_test.shape}")

## 4. Models — define, train, validate, test

Three models, increasing capacity:

- **Logistic Regression** — a linear baseline (one dense softmax layer).
- **MLP** — two hidden Dense layers, ReLU.
- **CNN** — two Conv1D + AvgPool blocks followed by a small dense head.

All three trained with Adam + sparse categorical cross-entropy, validation set monitored via `validation_data`. We collect each model's validation and test accuracy into `results` for the comparison plot at the end.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

INPUT_LEN = X_train.shape[1]
N_CLASSES = 9
EPOCHS    = 30
BATCH     = 256

results = {}
histories = {}

def evaluate(model, X, y):
    loss, acc = model.evaluate(X, y, batch_size=512, verbose=0)
    return acc

### 4a. Logistic Regression

In [ ]:
logreg = keras.Sequential([
    layers.Input(shape=(INPUT_LEN,)),
    layers.Dense(N_CLASSES, activation='softmax'),
], name='logreg')
logreg.compile(optimizer=keras.optimizers.Adam(1e-3),
               loss='sparse_categorical_crossentropy', metrics=['accuracy'])
logreg.summary()

hist = logreg.fit(X_train, y_train, validation_data=(X_val, y_val),
                  epochs=EPOCHS, batch_size=BATCH, verbose=2)
histories['Logistic Regression'] = hist
results['Logistic Regression'] = {
    'val_acc':  evaluate(logreg, X_val,  y_val),
    'test_acc': evaluate(logreg, X_test, y_test),
}
print(results['Logistic Regression'])

### 4b. MLP

In [ ]:
mlp = keras.Sequential([
    layers.Input(shape=(INPUT_LEN,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dense(N_CLASSES, activation='softmax'),
], name='mlp')
mlp.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss='sparse_categorical_crossentropy', metrics=['accuracy'])
mlp.summary()

hist = mlp.fit(X_train, y_train, validation_data=(X_val, y_val),
               epochs=EPOCHS, batch_size=BATCH, verbose=2)
histories['MLP'] = hist
results['MLP'] = {
    'val_acc':  evaluate(mlp, X_val,  y_val),
    'test_acc': evaluate(mlp, X_test, y_test),
}
print(results['MLP'])

### 4c. CNN

In [ ]:
X_train_c = X_train[..., None]
X_val_c   = X_val[..., None]
X_test_c  = X_test[..., None]

cnn = keras.Sequential([
    layers.Input(shape=(INPUT_LEN, 1)),
    layers.Conv1D(32, kernel_size=11, activation='relu', padding='same'),
    layers.AveragePooling1D(pool_size=2),
    layers.Conv1D(64, kernel_size=11, activation='relu', padding='same'),
    layers.AveragePooling1D(pool_size=2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(N_CLASSES, activation='softmax'),
], name='cnn')
cnn.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn.summary()

hist = cnn.fit(X_train_c, y_train, validation_data=(X_val_c, y_val),
               epochs=EPOCHS, batch_size=BATCH, verbose=2)
histories['CNN'] = hist
results['CNN'] = {
    'val_acc':  evaluate(cnn, X_val_c,  y_val),
    'test_acc': evaluate(cnn, X_test_c, y_test),
}
print(results['CNN'])

## 5. Results comparison

Validation vs. test accuracy side-by-side, plus a learning-curve plot per model.

In [ ]:
import pandas as pd
from pathlib import Path
Path('results').mkdir(exist_ok=True)

df = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'model'})
df.to_csv('results/comparison.csv', index=False)
print(df.to_string(index=False))

ax = df.plot(x='model', y=['val_acc', 'test_acc'], kind='bar', figsize=(7, 4), rot=0)
ax.set_ylabel('Accuracy'); ax.set_title('Model comparison — Hamming-weight classification')
for c in ax.containers:
    ax.bar_label(c, fmt='%.3f', padding=2, fontsize=9)
ax.set_ylim(0, max(df[['val_acc','test_acc']].max()) * 1.15)
plt.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig('results/comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for ax, (name, hist) in zip(axes, histories.items()):
    ax.plot(hist.history['accuracy'],     label='train')
    ax.plot(hist.history['val_accuracy'], label='val')
    ax.set_title(name); ax.set_xlabel('epoch'); ax.grid(alpha=0.3); ax.legend()
axes[0].set_ylabel('Accuracy')
plt.suptitle('Learning curves', y=1.02); plt.tight_layout()
plt.savefig('results/learning_curves.png', dpi=120, bbox_inches='tight')
plt.show()